# Notebook 07: MLflow Classification — Failure Prediction
## Purpose: Train + compare 3 models to predict machine failure in next 30 cycles
## Input:  workspace.predictive_maintenance.gold_ml_input
## Output: Best model registered in MLflow Model Registry (Production stage)

## ML Task: Binary Classification
## Target:  fail_30 (1 = machine will fail within 30 cycles, 0 = safe)
## Models:  Logistic Regression vs Random Forest vs XGBoost
## Metrics: F1-Score, AUC-ROC, Precision, Recall

## Why F1 and not Accuracy?
## Class imbalance — most cycles are non-failure.
## Accuracy would be misleading. F1 balances precision + recall.

In [0]:
%pip install xgboost shap matplotlib seaborn scikit-learn

dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                             recall_score, confusion_matrix,
                             classification_report)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

# Load ML input table
df = spark.table("workspace.predictive_maintenance.gold_ml_input")
print(f" Loaded: {df.count():,} rows | {len(df.columns)} columns")

In [0]:
# Define feature columns — exclude targets + metadata
EXCLUDE_COLS = [
    "unit_id", "cycle", "RUL", "fail_30", "fail_15",
    "source_dataset", "setting_1", "setting_2", "setting_3"
]

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]
TARGET_COL = "fail_30"

print(f"Feature columns: {len(FEATURE_COLS)}")
print(f"Target column:   {TARGET_COL}")

# Convert to pandas for sklearn
pdf = df.select(FEATURE_COLS + [TARGET_COL]).toPandas()

# Check class balance
pos = pdf[TARGET_COL].sum()
total = len(pdf)
print(f"\n=== CLASS BALANCE ===")
print(f"Positive (fail=1): {pos:,} ({pos/total*100:.1f}%)")
print(f"Negative (fail=0): {total-pos:,} ({(total-pos)/total*100:.1f}%)")

In [0]:
X = pdf[FEATURE_COLS].fillna(0)
y = pdf[TARGET_COL]

# 80/20 split — stratified to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train set: {X_train.shape[0]:,} rows")
print(f"Test set:  {X_test.shape[0]:,} rows")
print(f"Features:  {X_train.shape[1]}")
print(f"Stratified split — class balance preserved")

In [0]:
def log_confusion_matrix(y_true, y_pred, model_name):
    """Save confusion matrix as PNG and return path"""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Failure', 'Failure'],
                yticklabels=['No Failure', 'Failure'],
                ax=ax)
    ax.set_title(f'{model_name} — Confusion Matrix', fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    plt.tight_layout()
    path = f"/tmp/{model_name.replace(' ', '_')}_cm.png"
    plt.savefig(path)
    plt.show()
    plt.close()
    return path

def evaluate_model(model, X_test, y_test, model_name, use_scaled=False):
    """Evaluate and print all metrics"""
    X = X_test_scaled if use_scaled else X_test
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    f1        = f1_score(y_test, y_pred)
    auc       = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)

    print(f"\n=== {model_name} RESULTS ===")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")

    return {
        "f1": f1, "auc": auc,
        "precision": precision, "recall": recall,
        "y_pred": y_pred, "y_prob": y_prob
    }

print("Helper functions defined")

In [0]:
# Set MLflow experiment
mlflow.set_experiment("/Shared/PredMaint-Classification")

with mlflow.start_run(run_name="Logistic_Regression") as run:

    # Train
    lr = LogisticRegression(C=1.0, solver='lbfgs',
                            max_iter=1000, random_state=42)
    lr.fit(X_train_scaled, y_train)

    # Evaluate
    metrics = evaluate_model(lr, X_test, y_test,
                             "Logistic Regression", use_scaled=True)

    # Log params
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("C", 1.0)
    mlflow.log_param("solver", "lbfgs")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("features_count", len(FEATURE_COLS))

    # Log metrics
    mlflow.log_metric("f1_score",  metrics["f1"])
    mlflow.log_metric("auc_roc",   metrics["auc"])
    mlflow.log_metric("precision", metrics["precision"])
    mlflow.log_metric("recall",    metrics["recall"])

    # Log confusion matrix
    cm_path = log_confusion_matrix(
        y_test, metrics["y_pred"], "Logistic_Regression"
    )
    mlflow.log_artifact(cm_path)

    # Log model
    mlflow.sklearn.log_model(lr, "logistic_regression_model")

    lr_run_id = run.info.run_id
    print(f"\nRun 1 logged. Run ID: {lr_run_id}")
    print(f"   F1={metrics['f1']:.4f} | AUC={metrics['auc']:.4f}")

In [0]:
with mlflow.start_run(run_name="Random_Forest") as run:

    # Train
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    # Evaluate
    metrics = evaluate_model(rf, X_test, y_test, "Random Forest")

    # Log params
    mlflow.log_param("model_type",   "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth",    10)
    mlflow.log_param("features_count", len(FEATURE_COLS))

    # Log metrics
    mlflow.log_metric("f1_score",  metrics["f1"])
    mlflow.log_metric("auc_roc",   metrics["auc"])
    mlflow.log_metric("precision", metrics["precision"])
    mlflow.log_metric("recall",    metrics["recall"])

    # Log confusion matrix
    cm_path = log_confusion_matrix(
        y_test, metrics["y_pred"], "Random_Forest"
    )
    mlflow.log_artifact(cm_path)

    # Log model
    mlflow.sklearn.log_model(rf, "random_forest_model")

    rf_run_id = run.info.run_id
    print(f"\nRun 2 logged. Run ID: {rf_run_id}")
    print(f"   F1={metrics['f1']:.4f} | AUC={metrics['auc']:.4f}")

In [0]:
with mlflow.start_run(run_name="XGBoost_Classifier") as run:

    # Train
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # Evaluate
    metrics = evaluate_model(xgb_model, X_test, y_test, "XGBoost")

    # Log params
    mlflow.log_param("model_type",      "XGBoost")
    mlflow.log_param("n_estimators",    200)
    mlflow.log_param("max_depth",       6)
    mlflow.log_param("learning_rate",   0.1)
    mlflow.log_param("subsample",       0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("features_count",  len(FEATURE_COLS))

    # Log metrics
    mlflow.log_metric("f1_score",  metrics["f1"])
    mlflow.log_metric("auc_roc",   metrics["auc"])
    mlflow.log_metric("precision", metrics["precision"])
    mlflow.log_metric("recall",    metrics["recall"])

    # Log confusion matrix
    cm_path = log_confusion_matrix(
        y_test, metrics["y_pred"], "XGBoost"
    )
    mlflow.log_artifact(cm_path)

    # Log model
    mlflow.xgboost.log_model(xgb_model, "xgboost_model")

    xgb_run_id = run.info.run_id
    xgb_f1     = metrics["f1"]
    xgb_auc    = metrics["auc"]
    print(f"\nRun 3 logged. Run ID: {xgb_run_id}")
    print(f"   F1={metrics['f1']:.4f} | AUC={metrics['auc']:.4f}")

In [0]:
with mlflow.start_run(run_name="XGBoost_SHAP_Explainability") as run:

    print("Generating SHAP values — this takes 2-3 minutes...")

    # SHAP explainer
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test.iloc[:500])

    # Plot 1: Global feature importance
    fig1, ax1 = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values,
        X_test.iloc[:500],
        plot_type="bar",
        show=False,
        max_display=20
    )
    plt.title("SHAP Global Feature Importance — Top 20 Features",
              fontweight='bold')
    plt.tight_layout()
    shap_global_path = "/tmp/shap_global_importance.png"
    plt.savefig(shap_global_path, bbox_inches='tight')
    plt.show()
    plt.close()

    # Plot 2: SHAP summary dot plot
    fig2, ax2 = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values,
        X_test.iloc[:500],
        show=False,
        max_display=20
    )
    plt.title("SHAP Summary Plot — Feature Impact on Failure Prediction",
              fontweight='bold')
    plt.tight_layout()
    shap_summary_path = "/tmp/shap_summary.png"
    plt.savefig(shap_summary_path, bbox_inches='tight')
    plt.show()
    plt.close()

    # Plot 3: Waterfall for single machine
    fig3 = plt.figure(figsize=(10, 6))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[0],
            base_values=explainer.expected_value,
            data=X_test.iloc[0],
            feature_names=FEATURE_COLS
        ),
        show=False,
        max_display=15
    )
    plt.title("SHAP Waterfall — Single Machine Failure Explanation",
              fontweight='bold')
    plt.tight_layout()
    shap_waterfall_path = "/tmp/shap_waterfall.png"
    plt.savefig(shap_waterfall_path, bbox_inches='tight')
    plt.show()
    plt.close()

    # Log all SHAP artifacts
    mlflow.log_artifact(shap_global_path)
    mlflow.log_artifact(shap_summary_path)
    mlflow.log_artifact(shap_waterfall_path)
    mlflow.log_param("model_type", "XGBoost_SHAP")
    mlflow.log_metric("f1_score", xgb_f1)
    mlflow.log_metric("auc_roc",  xgb_auc)

    print("SHAP plots generated and logged to MLflow!")
    print("   Check: MLflow UI → Experiments → Artifacts tab")

In [0]:
print("=" * 60)
print("MODEL COMPARISON — PredMaint-Classification")
print("=" * 60)
print(f"{'Model':<25} {'F1':>8} {'AUC':>8}")
print("-" * 60)

# Recompute for display
models = {
    "Logistic Regression": (lr,  True),
    "Random Forest":       (rf,  False),
    "XGBoost":            (xgb_model, False),
}

best_f1 = 0
best_model_name = ""

for name, (model, scaled) in models.items():
    X = X_test_scaled if scaled else X_test
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    print(f"{name:<25} {f1:>8.4f} {auc:>8.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name

print("=" * 60)
print(f"Best Model: {best_model_name} (F1={best_f1:.4f})")
print("=" * 60)

In [0]:
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature

client = MlflowClient()

# Get the XGBoost run
runs = client.search_runs(
    experiment_ids=[
        mlflow.get_experiment_by_name(
            "/Shared/PredMaint-Classification"
        ).experiment_id
    ],
    filter_string="tags.mlflow.runName = 'XGBoost_Classifier'",
    order_by=["metrics.f1_score DESC"],
    max_results=1
)

best_run = runs[0]
best_run_id = best_run.info.run_id

print(f"✅ Best run ID: {best_run_id}")
print(f"   F1:  {best_run.data.metrics['f1_score']:.4f}")
print(f"   AUC: {best_run.data.metrics['auc_roc']:.4f}")

# Re-log model WITH signature before registering
with mlflow.start_run(run_id=best_run_id):
    signature = infer_signature(
        X_train,
        xgb_model.predict(X_train)
    )
    mlflow.xgboost.log_model(
        xgb_model,
        "xgboost_model",
        signature=signature,
        input_example=X_train.iloc[:5]
    )

# Register model
model_uri = f"runs:/{best_run_id}/xgboost_model"
model_details = mlflow.register_model(
    model_uri=model_uri,
    name="PredMaint-Classifier"
)

print(f"\n✅ Model registered: PredMaint-Classifier")
print(f"   Version: {model_details.version}")

In [0]:
import time
time.sleep(5)

# Unity Catalog uses aliases instead of stages
# "champion" alias = equivalent to "Production" stage
client.set_registered_model_alias(
    name="workspace.default.predmaint-classifier",
    alias="champion",
    version=model_details.version
)

print("PredMaint-Classifier → 'champion' alias set!")
print("   (Unity Catalog uses aliases instead of stages)")
print("\n   Check: MLflow UI → Models → predmaint-classifier")
print("   You should see 'champion' alias on the model")
print("\n   THIS IS YOUR VIDEO DEMO MOMENT 🎬=")
print("   Say: 'Our best XGBoost model is promoted to champion")
print("   — the production-ready version for inference'")

In [0]:
print("=" * 60)
print("NOTEBOOK 07 COMPLETE — FAILURE CLASSIFICATION")
print("=" * 60)
print(f" MLflow Experiment:  PredMaint-Classification")
print(f" Runs logged:        3 (LR + RF + XGBoost)")
print(f" SHAP plots:         3 artifacts logged")
print(f" Best model:         XGBoost (F1={xgb_f1:.4f})")
print(f" Model Registry:     PredMaint-Classifier")
print(f" Stage:              Production")
print("=" * 60)
print("READY FOR NOTEBOOK 08 — RUL Regression")
print("=" * 60)